<span style="font-weight:bold; font-size: 3rem; color:#333;">- Part 02: Feature pipeline for Train Delay Data (Two-Stage Model)</span>

## 🗒️ Overview

This notebook downloads new data and ingests it into hopsworks feature groups.

It performs the following steps:

1. 


### 📝 Imports

In [8]:
# top of notebook
from features import build_features, add_calendar_features, add_train_lag_features, \
     add_station_network_state_features, detect_trigger_time, \
     add_reactive_early_dynamics, add_weather_rolling_features_if_present


In [9]:
import os
import datetime
import pandas as pd
import requests
import hopsworks_utils
import datetime as dt
from dotenv import load_dotenv
import hopsworks
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

True

## 📡 Connect to Hopsworks Feature Store

In [10]:
#### it need to get fixed

try:
    project = hopsworks_utils.HopsworksInterface()
    print("Hopsworks login OK")
except Exception as e:
    project = None
    print("Hopsworks not configured / login failed (OK). Proceeding without it.")
    print("Reason:", repr(e))

#train_feature_df = project.get("train_stop_events_labeled")

#uncoment the below line when weather features are stored
#weather_df = project.get("weather_features") 


HOPSWORKS_API_KEY exists: True
HOPSWORKS_API_KEY length: 81
2026-01-05 17:04:42,554 INFO: Closing external client and cleaning up certificates.
2026-01-05 17:04:42,559 INFO: Connection closed.
2026-01-05 17:04:42,563 INFO: Initializing external client
2026-01-05 17:04:42,564 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-01-05 17:04:44,016 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/2182
Hopsworks login OK


In [11]:
# Paths
CANONICAL_PATH = os.getenv("CANONICAL_PATH", "data/train_stop_events_labeled.parquet")
#data/train_stop_events_labeled.parquet
#out_path = "data/train_stop_events_labeled.parquet"
#out_path_weather = "data/weather_features.parquet"



OUT_DIR = os.getenv("OUT_DIR", "data/feature_pipeline_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# Label settings (must match Part 01)
HORIZON_MIN = int(os.getenv("HORIZON_MIN", "60"))
DELAY_THRESHOLD_MIN = int(os.getenv("DELAY_THRESHOLD_MIN", "10"))

# Rolling windows for network-state features
ROLL_WINDOWS_MIN = [30, 60]   # minutes
WEATHER_ROLL_WINDOWS_H = [3, 6]  # hours (only used if weather columns exist)

# Split ratios (time-based, by unique dates)
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9


In [12]:
if not os.path.exists(CANONICAL_PATH):
    raise FileNotFoundError(
        f"Could not find canonical dataset at {CANONICAL_PATH}. "
        "Run Part 01 and make sure it saved train_stop_events_labeled.parquet."
    )

df = pd.read_parquet(CANONICAL_PATH)

print("Loaded:", CANONICAL_PATH)
print("Shape:", df.shape)
display(df.head())


Loaded: data/train_stop_events_labeled.parquet
Shape: (76076, 33)


,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,train_run_id,final_delay_min,additional_delay_min,y_delay_within_horizon,temperature_2m,precipitation,rain,snowfall,windspeed_10m,weather_time
0,1500adde-075d-66fb-08de-3de2e9c51bd7,Avgang,2983,2026-01-02 00:00:00,2026-01-02 00:00:00+01:00,NaT,2026-01-02 00:00:00+01:00,2026-01-02 00:00:00+01:00,Mr,0.0,...,2983_2026-01-02,0.0,0.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
1,1500adde-075d-66fb-08de-3de2c860a960,Avgang,2477,2026-01-02 00:03:00,2026-01-02 00:03:00+01:00,NaT,2026-01-02 00:03:00+01:00,2026-01-02 00:03:00+01:00,Söc,0.0,...,2477_2026-01-02,-2.0,-2.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
2,1500adde-075d-66fb-08de-3de2e9c6a56c,Avgang,2983,2026-01-02 00:04:00,2026-01-02 00:04:00+01:00,NaT,2026-01-02 00:05:00+01:00,2026-01-02 00:05:00+01:00,Rs,1.0,...,2983_2026-01-02,0.0,-1.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
3,1500adde-075d-66fb-08de-3de2e9c6a56b,Ankomst,2983,2026-01-02 00:04:00,2026-01-02 00:04:00+01:00,NaT,2026-01-02 00:04:00+01:00,2026-01-02 00:04:00+01:00,Rs,0.0,...,2983_2026-01-02,0.0,0.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02
4,1500adde-075d-66fb-08de-3de314d64fcf,Avgang,7879,2026-01-02 00:05:00,2026-01-02 00:05:00+01:00,NaT,2026-01-02 00:05:00+01:00,2026-01-02 00:05:00+01:00,Arnn,0.0,...,7879_2026-01-02,1.0,1.0,0,0.3,0.0,0.0,0.0,6.9,2026-01-02


In [13]:
# Ensure required columns exist
required_cols = ["event_time", "station_code", "train_id", "delay_min",
                 "y_delay_within_horizon", "final_delay_min", "additional_delay_min"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in canonical dataset: {missing}")

df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")
df = df.dropna(subset=["event_time", "station_code", "train_id"]).copy()

# Ensure a train-run key exists
if "train_run_id" not in df.columns:
    df["date"] = df["event_time"].dt.date
    df["train_run_id"] = df["train_id"].astype(str) + "_" + df["date"].astype(str)

# Sort for point-in-time computations
df = df.sort_values(["event_time", "station_code", "train_run_id"]).reset_index(drop=True)

# Normalize reason_code
if "reason_code" in df.columns:
    df["reason_code"] = df["reason_code"].astype("string")
else:
    df["reason_code"] = pd.Series([pd.NA]*len(df), dtype="string")

# --- RENAME STEP ADDED HERE ---
weather_rename_map = {
    "temperature_2m": "weather_temperature_2m",
    "precipitation": "weather_precipitation",
    "rain": "weather_rain",
    "snowfall": "weather_snowfall",
    "windspeed_10m": "weather_windspeed_10m"
}
existing_rename = {k: v for k, v in weather_rename_map.items() if k in df.columns}
if existing_rename:
    print(f"Renaming weather columns: {list(existing_rename.keys())}")
    df = df.rename(columns=existing_rename)
# ------------------------------

print("After cleaning:", df.shape)

Renaming weather columns: ['temperature_2m', 'precipitation', 'rain', 'snowfall', 'windspeed_10m']
After cleaning: (76076, 33)


### Feature engineering (no leakage)

In [14]:
# Apply feature engineering
print("⚡️ Engineering features using 'features.py'...")

df_feat = df.copy()

# 1. Calendar
df_feat = add_calendar_features(df_feat)

# 2. Lag features
df_feat = add_train_lag_features(df_feat)

# 3. Network State
df_feat = add_station_network_state_features(df_feat, windows_min=ROLL_WINDOWS_MIN)

# 4. Trigger Detection
df_feat = detect_trigger_time(df_feat)

# 5. Reactive Dynamics
df_feat = add_reactive_early_dynamics(df_feat)

# 6. Weather
df_feat = add_weather_rolling_features_if_present(df_feat, windows_h=WEATHER_ROLL_WINDOWS_H)

print("✅ Feature engineering complete.")
print("Feature table shape:", df_feat.shape)
display(df_feat.head())

⚡️ Engineering features using 'features.py'...
✅ Feature engineering complete.
Feature table shape: (76076, 57)


,event_time,ActivityId,ActivityType,train_id,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,weather_temperature_2m_rollmean_3h,weather_precipitation_rollmean_3h,weather_rain_rollmean_3h,weather_snowfall_rollmean_3h,weather_windspeed_10m_rollmean_3h,weather_temperature_2m_rollmean_6h,weather_precipitation_rollmean_6h,weather_rain_rollmean_6h,weather_snowfall_rollmean_6h,weather_windspeed_10m_rollmean_6h
0,2026-01-02 00:29:00,1500adde-075d-66fb-08de-3de279b4d082,Ankomst,10994,2026-01-02 00:29:00+01:00,2026-01-02 00:30:00+01:00,2026-01-02 00:30:00+01:00,2026-01-02 00:30:00+01:00,Arnc,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-01-02 00:29:00,1500adde-075d-66fb-08de-3de279b4d083,Avgang,10994,2026-01-02 00:29:00+01:00,2026-01-02 00:30:00+01:00,2026-01-02 00:32:00+01:00,2026-01-02 00:32:00+01:00,Arnc,3.0,...,0.300000,0.0,0.0,0.0,6.900000,0.300000,0.0,0.0,0.0,6.900000
2,2026-01-02 00:44:00,1500adde-075d-66fb-08de-3de2c0729363,Avgang,2285,2026-01-02 00:44:00+01:00,2026-01-02 00:47:00+01:00,2026-01-02 00:45:00+01:00,2026-01-02 00:45:00+01:00,Arnc,1.0,...,0.300000,0.0,0.0,0.0,6.900000,0.300000,0.0,0.0,0.0,6.900000
3,2026-01-02 00:44:00,1500adde-075d-66fb-08de-3de2c0729362,Ankomst,2285,2026-01-02 00:44:00+01:00,2026-01-02 00:45:00+01:00,2026-01-02 00:44:00+01:00,2026-01-02 00:44:00+01:00,Arnc,0.0,...,0.166667,0.0,0.0,0.0,6.266667,0.166667,0.0,0.0,0.0,6.266667
4,2026-01-02 04:44:00,1500adde-075d-66fb-08de-3de2b72354bf,Ankomst,2205,2026-01-02 04:44:00+01:00,NaT,2026-01-02 04:44:00+01:00,2026-01-02 04:44:00+01:00,Arnc,0.0,...,NaN,NaN,NaN,NaN,NaN,0.100000,0.0,0.0,0.0,5.950000


In [15]:
import os
import pandas as pd
import numpy as np
import json
import datetime as dt

# --- 0. PRE-REQUISITE: DEFINE FEATURE COLUMNS ---
# Define which columns are ID/Target/Future and should be excluded
ALL_KEYS = ["ActivityId", "train_id", "OperationalTrainNumber", "station_code", "event_time", "event_date", "train_run_id"]
TARGETS = ["y_delay_within_horizon", "final_delay_min", "additional_delay_min"]

# Columns to exclude from Predictive Model (future leakage or reactive-only)
PRED_EXCLUDE = set(TARGETS + ["final_delay_min", "additional_delay_min", 
                              "trigger_time", "min_since_trigger", "delay_at_trigger", 
                              "delay_slope_since_trigger", "is_first10m_after_trigger"])

# Columns to exclude from Reactive Model (just the predictive target)
REACT_EXCLUDE = set(["y_delay_within_horizon"])

# Calculate the lists of columns dynamically from df_feat
pred_cols = [c for c in df_feat.columns if c not in ALL_KEYS and c not in PRED_EXCLUDE]
react_cols = [c for c in df_feat.columns if c not in ALL_KEYS and c not in TARGETS and c not in REACT_EXCLUDE]

print(f"Features detected: {len(pred_cols)} Predictive, {len(react_cols)} Reactive")


# --- 1. SPLIT LOGIC ---
df_feat["event_date"] = df_feat["event_time"].dt.date
dates = sorted(df_feat["event_date"].unique())
n = len(dates)

if n >= 3:
    n_val = max(1, int(np.floor(n * 0.15)))  
    n_test = max(1, int(np.floor(n * 0.15))) 
    n_train = n - n_val - n_test
else:
    n_train = n
    n_val = 0
    n_test = 0

train_dates = set(dates[:n_train])
val_dates   = set(dates[n_train:n_train+n_val])
test_dates  = set(dates[n_train+n_val:])

print(f"Refined Split: Train={len(train_dates)}d, Val={len(val_dates)}d, Test={len(test_dates)}d")

# --- 2. RE-SLICE DATAFRAMES ---
df_train = df_feat[df_feat["event_date"].isin(train_dates)].copy()
df_val   = df_feat[df_feat["event_date"].isin(val_dates)].copy()
df_test  = df_feat[df_feat["event_date"].isin(test_dates)].copy()

# Reactive Slices (Filter for delay >= threshold)
df_react_train = df_train[df_train["delay_min"] >= DELAY_THRESHOLD_MIN].copy()
df_react_val   = df_val[df_val["delay_min"] >= DELAY_THRESHOLD_MIN].copy()
df_react_test  = df_test[df_test["delay_min"] >= DELAY_THRESHOLD_MIN].copy()

# --- 3. REGENERATE X AND y ---
# Predictive Targets
X_pred_train = df_train[pred_cols]
y_pred_train = df_train["y_delay_within_horizon"]

X_pred_val   = df_val[pred_cols]
y_pred_val   = df_val["y_delay_within_horizon"]

X_pred_test  = df_test[pred_cols]
y_pred_test  = df_test["y_delay_within_horizon"]

# Reactive Targets
X_react_train = df_react_train[react_cols]
y_react_train = df_react_train["additional_delay_min"]

X_react_val   = df_react_val[react_cols]
y_react_val   = df_react_val["additional_delay_min"]

X_react_test  = df_react_test[react_cols]
y_react_test  = df_react_test["additional_delay_min"]

# --- 4. PACK AND SAVE ---
# Keys to keep in the final output for joining
SAVE_KEYS = ["train_run_id", "event_time", "station_code", "train_id"]

pred_train_path = os.path.join(OUT_DIR, "pred_train.parquet")
pred_val_path   = os.path.join(OUT_DIR, "pred_val.parquet")
pred_test_path  = os.path.join(OUT_DIR, "pred_test.parquet")

react_train_path = os.path.join(OUT_DIR, "react_train.parquet")
react_val_path   = os.path.join(OUT_DIR, "react_val.parquet")
react_test_path  = os.path.join(OUT_DIR, "react_test.parquet")

def pack(df_split, X, y, task: str) -> pd.DataFrame:
    if df_split.empty:
        return pd.DataFrame()
        
    actual_keys = [k for k in SAVE_KEYS if k in df_split.columns]
    packed = df_split[actual_keys].copy().reset_index(drop=True)
    
    if isinstance(X, pd.DataFrame):
        X = X.reset_index(drop=True)
        packed = pd.concat([packed, X], axis=1)
    else:
        packed = packed.join(pd.DataFrame(X, columns=pred_cols if task=="pred" else react_cols))
        
    # Correctly name the target column
    if task == "pred":
        packed["y_delay_within_horizon"] = y.values
    elif task == "react":
        packed["additional_delay_min"] = y.values
    else:
        packed[f"y_{task}"] = y.values
    
    return packed

def save_parquet(df, path):
    if not df.empty:
        df.to_parquet(path, index=False)
        print(f"✅ Saved: {path} ({len(df)} rows)")
    else:
        print(f"⚠️ Empty split, saving schema only: {path}")
        df.to_parquet(path, index=False)

# Save
save_parquet(pack(df_train, X_pred_train, y_pred_train, "pred"), pred_train_path)
save_parquet(pack(df_val,   X_pred_val,   y_pred_val,   "pred"), pred_val_path)
save_parquet(pack(df_test,  X_pred_test,  y_pred_test,  "pred"), pred_test_path)

save_parquet(pack(df_react_train, X_react_train, y_react_train, "react"), react_train_path)
save_parquet(pack(df_react_val,   X_react_val,   y_react_val,   "react"), react_val_path)
save_parquet(pack(df_react_test,  X_react_test,  y_react_test,  "react"), react_test_path)

# Metadata (Includes feature names for Part 4!)
metadata = {
    "created_utc": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "splits": {
        "train_dates": [str(d) for d in sorted(train_dates)],
        "val_dates": [str(d) for d in sorted(val_dates)],
        "test_dates": [str(d) for d in sorted(test_dates)],
    },
    "pred_feature_columns": pred_cols,   # <--- Critical for Part 4
    "react_feature_columns": react_cols  # <--- Critical for Part 4
}
meta_path = os.path.join(OUT_DIR, "feature_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Metadata saved to {meta_path}")

Features detected: 45 Predictive, 49 Reactive
Refined Split: Train=2d, Val=1d, Test=1d
✅ Saved: data/feature_pipeline_outputs/pred_train.parquet (41698 rows)
✅ Saved: data/feature_pipeline_outputs/pred_val.parquet (17917 rows)
✅ Saved: data/feature_pipeline_outputs/pred_test.parquet (16461 rows)
✅ Saved: data/feature_pipeline_outputs/react_train.parquet (2113 rows)
✅ Saved: data/feature_pipeline_outputs/react_val.parquet (2316 rows)
✅ Saved: data/feature_pipeline_outputs/react_test.parquet (4665 rows)
✅ Metadata saved to data/feature_pipeline_outputs/feature_metadata.json


In [16]:
# Verify weather columns are in the list
weather_cols_saved = [c for c in pred_cols if "weather" in c]
print(f"n📊 Verification: {len(weather_cols_saved)} weather features included in predictive model.")
if len(weather_cols_saved) > 0:
    print("Example features:", weather_cols_saved[:3])
else:
    print("⚠️ WARNING: No weather features detected in the final list!")


n📊 Verification: 16 weather features included in predictive model.
Example features: ['weather_temperature_2m', 'weather_precipitation', 'weather_rain']


### ✂️ Time-based train/val/test split (no leakage)

### 🧩 Build predictive vs reactive feature matrices

### 💾 Save outputs + feature metadata